# Automated Pipeline Search with TPOT (Tree-based Pipeline Optimization Tool)

---

### Methodological Framework & Search Architecture:

1. **Explicit Evolutionary Hyperparameters:**
   - **Population Size:** $N = 40$ (number of candidate pipelines per generation).
   - **Number of Generations:** $G = 15$ to $20$ iterations of evolutionary pipeline synthesis.
   - **Offspring Generation:** Mutation rate = 0.90, Crossover rate = 0.10.
   - **Optimization Metric:** `scoring='roc_auc'` (maximizing area under the ROC curve for class-imbalanced classification).
   - **Internal Validation Strategy:** Stratified 5-Fold Cross-Validation on the training partition.

2. **Computational Convergence without Premature Truncation:**
   - The search is governed by generation count ($G=15$) and early stopping criteria (`early_stop=5`), guaranteeing thorough exploration across feature transformers, selectors, and classifiers without premature termination.

3. **Strict Zero-Leakage Data Partitioning:**
   - Evaluated on a strictly isolated 20% holdout test set (`X_test, y_test`) partitioned with stratification before initiating the genetic search.


In [4]:
%pip install "setuptools<81"

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix, roc_curve, auc)
from tpot import TPOTClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment loaded for TPOT AutoML search.")


Environment loaded for TPOT AutoML search.


## 1. Data Ingestion & Preprocessing

Loading `Datasets/train_processed.csv`.
We encode categorical variables and establish the stratified 80% Train / 20% Holdout Test split.


In [6]:
df = pd.read_csv("/home/ander/Documentos/Universidad/ML/proyecto/Datasets/train_processed.csv")
X = df.drop(columns=["Exited"])
y = df["Exited"].astype(int)

# One-Hot Encoding for categorical features (Geography, Gender)
categorical_columns = ['Geography', 'Gender']
X_encoded = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

# 80/20 Stratified Partition
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"X_train shape: {X_train.shape} (Churn: {y_train.mean()*100:.2f}%)")
print(f"X_test shape:  {X_test.shape} (Churn: {y_test.mean()*100:.2f}%)")


X_train shape: (12000, 11) (Churn: 20.38%)
X_test shape:  (3000, 11) (Churn: 20.40%)


## 2. TPOT Evolutionary Search Setup

We explicitly define and document all evolutionary parameters:
- `generations`: 15 (evolutionary rounds)
- `population_size`: 40 (individuals per generation)
- `mutation_rate`: 0.90
- `crossover_rate`: 0.10
- `scoring`: `'roc_auc'` (maximizing discrimination on imbalanced target)
- `cv`: Stratified 5-Fold Cross-Validation
- `n_jobs`: -1 (parallel CPU cores)

*You can run this cell directly on your machine.*


In [7]:
# Configure Stratified 5-Fold CV for internal pipeline evaluation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Instantiate TPOT Classifier with fully documented hyperparameters
tpot_model = TPOTClassifier(
    generations=15,               # Number of evolutionary generations
    population_size=40,           # Number of pipelines retained each generation
    mutation_rate=0.90,           # Probability of mutation
    crossover_rate=0.10,          # Probability of crossover
    scoring='roc_auc',            # Optimize ROC-AUC for imbalanced banking churn
    cv=cv_strategy,               # 5-Fold Stratified CV on X_train
    random_state=RANDOM_STATE,    # Reproducibility seed
    verbosity=2,                  # Detailed logging per generation
    n_jobs=-1,                    # Use all available CPU cores
    max_time_mins=None            # No artificial time truncation (allows full generational convergence)
)

print("Starting TPOT evolutionary search on X_train...")
# Execute genetic search
tpot_model.fit(X_train, y_train)

print("\nTPOT search completed!")
print("Best Pipeline internal CV ROC-AUC:", tpot_model.score(X_train, y_train))
print("\nBest Pipeline Structure:")
print(tpot_model.fitted_pipeline_)


Starting TPOT evolutionary search on X_train...


Version 0.12.2 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


Optimization Progress:   0%|          | 0/640 [00:00<?, ?pipeline/s]

/home/ander/miniconda3/envs/Churn/lib/python3.11/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/ander/miniconda3/envs/Churn/lib/python3.11/site-packages/stopit/__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/ander/miniconda3/envs/Churn/lib/python3.11/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")

                                                                             
Generation 1 - Current best internal CV score: 0.9333404628260844
                                                                                
Generation 2 - Current best internal CV score: 0.9341334846554389
                                                                                
Generation 3 - Current best internal CV score: 0.9349870514179447
                                                                              
Generation 4 - Current best internal CV score: 0.9349870514179447
                                                                              
Generation 5 - Current best internal CV score: 0.9349870514179447
                                                                              
Generation 6 - Current best internal CV score: 0.9349870514179447
                                                                              
Generation 7 - Current best internal CV score: 0

## 3. Holdout Test Set Evaluation & Export

We evaluate the optimal pipeline discovered by TPOT on the untouched holdout test set (`X_test, y_test`).


In [8]:
# Predict on holdout test set
y_pred = tpot_model.predict(X_test)
y_pred_proba = tpot_model.predict_proba(X_test)[:, 1]

# Metrics
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
rec = recall_score(y_test, y_pred)
spec = tn / (tn + fp)
gmean = np.sqrt(rec * spec)

print("="*60)
print("TPOT OPTIMAL PIPELINE - HOLDOUT TEST SET PERFORMANCE")
print("="*60)
print(f"Accuracy:         {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision:        {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Sens.):   {rec:.4f}")
print(f"Specificity:      {spec:.4f}")
print(f"F1-Score:         {f1_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:     {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"G-Mean:           {gmean:.4f}")
print("="*60)

# Export Python script of best pipeline
tpot_model.export("tpot_best_pipeline.py")
print("Exported best pipeline code to 'tpot_best_pipeline.py'.")


TPOT OPTIMAL PIPELINE - HOLDOUT TEST SET PERFORMANCE
Accuracy:         0.8953
Precision:        0.7833
Recall (Sens.):   0.6732
Specificity:      0.9523
F1-Score:         0.7241
Test ROC-AUC:     0.9323
G-Mean:           0.8007
Exported best pipeline code to 'tpot_best_pipeline.py'.


In [10]:
# Generate Kaggle submission
test_kaggle = pd.read_csv("/home/ander/Documentos/Universidad/ML/proyecto/Datasets/test.csv")
test_encoded = pd.get_dummies(test_kaggle.drop(columns=['id', 'CustomerId', 'Surname']), columns=categorical_columns, drop_first=True)

# Ensure columns match X_train
for col in X_train.columns:
    if col not in test_encoded.columns:
        test_encoded[col] = 0
test_encoded = test_encoded[X_train.columns]

tpot_test_preds = tpot_model.predict_proba(test_encoded)[:, 1]

os.makedirs("Submissions", exist_ok=True)
sub_path = "Submissions/tpot_submission.csv"
pd.DataFrame({'id': test_kaggle['id'], 'Exited': tpot_test_preds}).to_csv(sub_path, index=False)
print(f"TPOT Kaggle submission saved to '{sub_path}'.")


TPOT Kaggle submission saved to 'Submissions/tpot_submission.csv'.
